# 3 Ingeniería de Datos con Pandas
## Importación de librerías

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import csv

## Carga del dataset

In [ ]:
''' Al haber detectado antes el problema con las comillas,
    la carga del dataset se hace ignorandolas como separadores '''
df = pd.read_csv(
    "../data/ventas_techventas.csv",
    quoting=csv.QUOTE_NONE,
    encoding="latin1"
)

df.head()

## Información general

In [ ]:
df.info()

## Observaciones de la estructura del dataset

El dataset contiene **60 registros** y **11 columnas** relacionadas con pedidos, clientes, productos y ventas de TechVentas S.A.S.

Se identifican tres tipos principales de datos:

- **Variables categóricas (object):** `order_id`, `cliente_id`, `cliente_nombre`, `region`, `producto`, `categoria` y `vendedor`.
- **Variables numéricas enteras (int64):** `cantidad`.
- **Variables numéricas decimales (float64):** `precio_unitario` y `descuento`.

Durante la exploración inicial se encontró un posible problema de calidad de datos en la columna **producto**, ya que presenta **59 valores no nulos de 60 registros**, lo que indica la existencia de un valor faltante que deberá ser tratado durante la fase de limpieza.

Las columnas numéricas (`cantidad`, `precio_unitario` y `descuento`) no presentan valores nulos según la información mostrada por `info()`. Sin embargo, es necesario realizar validaciones adicionales para detectar posibles valores inconsistentes, como cantidades negativas o descuentos fuera de rango.

La columna `fecha` aparece como tipo `object`, es necesario usar el parámetro `parse_dates` para convertirla a un formato de fecha adecuado y facilitar posteriores análisis temporales y agrupaciones mensuales.

En general, el dataset presenta una estructura consistente y una baja cantidad de datos faltantes, lo que facilita las tareas de limpieza y transformación posteriores. Esto se debe a que en la carga de datos se ignora las comillas dobles que causaba el agrupamiento de información de varias columnas y dejando muchos valores nulos. 

## Detectar nulos con isna()

In [ ]:
problemas = df[df.isna().any(axis=1)]

problemas

## Eliminar/imputar

In [ ]:
''' Dentro de los nombres de productos solo existe uno con precio unitario = 350.0 y 
    que sea de la categoria Electronica, por lo tanto, se ubica el registro cuyo nombre 
    del producto es nulo y cumple con las características mencionadas y se imputa el 
    nombre del producto, en este caso "Monitor 27"  '''

df.loc[
    (df["producto"].isna()) &
    (df["precio_unitario"] == 350.0) &
    (df["categoria"] == "Electronica"),
    "producto"
] = "Monitor 27"

## luego del proceso Eliminar/imputar se descartaron 0 registros, puesto que el único registro con problemas fue recuperado usando la información disponible

##  Crear columnas ingreso_bruto, ingreso_neto y mes.

In [ ]:
df["ingreso_bruto"] = (
    df["cantidad"] *
    df["precio_unitario"]
)
df["ingreso_neto"] = (
    df["ingreso_bruto"] *
    (1 - df["descuento"])
)
# Normalizar fechas para poder crear mes

df["fecha"] = pd.to_datetime(
    df["fecha"],
    errors="coerce"
)

df = df[df["fecha"].notna()]
df["mes"] = df["fecha"].dt.to_period("M")

df[
    [
        "fecha",
        "cantidad",
        "precio_unitario",
        "ingreso_bruto",
        "ingreso_neto",
        "mes"
    ]
].head()

## Agrupación mensual

In [ ]:
resumen = (
    df.groupby("mes")
      .agg(
          total=("ingreso_neto", "sum"),
          n=("order_id", "count")
      )
      .reset_index()
)

resumen

## Merge con tabla de metas

In [ ]:
# Crear tabla de metas
metas = pd.DataFrame({
    "mes": resumen["mes"],
    "meta": [
        10000,
        10000,
        10000,
        10000,
        10000,
        10000
    ][:len(resumen)]
})

metas

## Merge

In [ ]:
cumplimiento = pd.merge(
    resumen,
    metas,
    on="mes",
    how="left"
)

cumplimiento

## Calcular porcentaje

In [ ]:
cumplimiento["porcentaje_cumplimiento"] = (
    cumplimiento["total"] /
    cumplimiento["meta"]
) * 100

cumplimiento

# Exportación SQLite
## Exportar

In [ ]:
engine = create_engine(
    "sqlite:///../output/techventas.db"
)
# Se convierte el campo mes a string para evitar problemas al exportar a SQL
df["mes"] = df["mes"].astype(str)
df.to_sql(
    "ventas",
    engine,
    if_exists="replace",
    index=False
)

print("Datos exportados correctamente")

##  verifica leyendo de vuelta con read_sql

In [ ]:
verificacion = pd.read_sql(
    "SELECT COUNT(*) FROM ventas",
    engine
)

verificacion

la verificación fue correcta, puesto que no se descartaron registros